# Demo 2: Prepare model-ready JSONL files for fine-tuning

This notebook converts cleaned positive MOF synthesis records and inferred negative synthesis records into model-ready JSONL files. It creates train/holdout splits, class maps, split summaries, and additional optional datasets used for model evaluation or ablation studies.

**Inputs expected in the same folder as this notebook**
- `mof_extraction_1_2_3_4_5_6.csv`
- `mof_extraction_failures_enum_1_2_3_4_5_6.csv`
- `Full.xlsx`

**Primary outputs**
- `out/mof_cls_train.jsonl`
- `out/mof_cls_holdout.jsonl`
- `out/mof_cls_class_map.json`
- `out/mof_cls_split_summary.json`

Run the notebook from top to bottom for the standard classification dataset. Later sections generate optional dataset variants.


## 1. Build the main positive/negative classification dataset

This section builds the standard P/N reaction-outcome dataset using cleaned literature successes and inferred failures.


In [ ]:

import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from math import ceil

# --------------------------
# Config
# --------------------------
POS_PATH = "mof_extraction_1_2_3_4_5_6.csv"
NEG_PATH = "mof_extraction_failures_enum_1_2_3_4_5_6.csv"
FULL_METADATA_PATH = "Full.xlsx"  # columns: DOI, Publication Year

OUT_DIR = Path("out")
TRAIN_OUT = OUT_DIR / "mof_cls_train.jsonl"
HOLDOUT_OUT = OUT_DIR / "mof_cls_holdout.jsonl"
CLASS_MAP_OUT = OUT_DIR / "mof_cls_class_map.json"
SUMMARY_OUT = OUT_DIR / "mof_cls_split_summary.json"

RNG_SEED = 42
HOLDOUT_FRAC = 0.10  # target holdout row fraction
HOLDOUT_CLUSTER_FRAC = 0.10  # target holdout cluster fraction

# "both" targets rows + clusters + label balance.
# "rows" targets only row count + label balance.
# "clusters" targets only cluster count + label balance.
HOLDOUT_TARGET_MODE = "both"
HOLDOUT_SEARCH_TRIALS = 800
YEAR_BINS = 4

# Default: this is the stratified split:
# metal + all linkers + all solvents.
# Use "precursor" for clusters in-domain holdout,
# or "element" if you want CdCl2 and Cd(NO3)2 grouped as Cd.
CLUSTER_METAL_MODE = "precursor"  # "precursor" or "element"
INCLUDE_MODULATOR_IN_CLUSTER = False

# Useful quality controls for classification fine-tuning.
DROP_INPUT_LABEL_CONFLICTS = True  # drop same input JSON appearing as both P and N
DEDUP_EXACT_INPUT_WITHIN_LABEL = False  # set True if duplicates dominate training
SHUFFLE_OUTPUT = True

SYSTEM_PROMPT = (
    """Act as an expert in reticular chemistry. You will receive reaction conditions as a JSON object with the fields: 
    metal_precursor, organic_linker, modulator, solvent, metal_concentration_mM, M_L_ratio, temperature_C, and time_h. 
    Based on these inputs, output exactly one uppercase label: 'P' if the conditions are likely to yield a crystalline 
    metal-organic framework under experimental conditions, or 'N' if not."""
)

LABEL_POS = "P"
LABEL_NEG = "N"

# --------------------------
# Helpers: cleaning/parsing
# --------------------------
EMPTY_TOKENS = {
    "", "nan", "none", "null", "na", "n/a", "not_reported",
    "not reported", "not-report", "unknown"
}

ELEMENT_NAMES = {
    "lithium": "Li", "sodium": "Na", "potassium": "K", "rubidium": "Rb", "cesium": "Cs",
    "magnesium": "Mg", "calcium": "Ca", "strontium": "Sr", "barium": "Ba",
    "scandium": "Sc", "yttrium": "Y", "lanthanum": "La", "cerium": "Ce",
    "praseodymium": "Pr", "neodymium": "Nd", "samarium": "Sm", "europium": "Eu",
    "gadolinium": "Gd", "terbium": "Tb", "dysprosium": "Dy", "holmium": "Ho",
    "erbium": "Er", "thulium": "Tm", "ytterbium": "Yb", "lutetium": "Lu",
    "titanium": "Ti", "zirconium": "Zr", "hafnium": "Hf", "vanadium": "V",
    "niobium": "Nb", "tantalum": "Ta", "chromium": "Cr", "molybdenum": "Mo",
    "tungsten": "W", "manganese": "Mn", "iron": "Fe", "cobalt": "Co", "nickel": "Ni",
    "copper": "Cu", "zinc": "Zn", "cadmium": "Cd", "mercury": "Hg", "aluminum": "Al",
    "gallium": "Ga", "indium": "In", "tin": "Sn", "lead": "Pb", "bismuth": "Bi",
    "silver": "Ag", "gold": "Au", "palladium": "Pd", "platinum": "Pt", "ruthenium": "Ru",
    "rhodium": "Rh", "iridium": "Ir", "osmium": "Os"
}

def clean_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = (
        s.replace("′", "'")
         .replace("’", "'")
         .replace("‘", "'")
         .replace("“", '"')
         .replace("”", '"')
         .replace("–", "-")
         .replace("—", "-")
    )
    s = s.strip('"').strip("'")
    s = re.sub(r"\s+", " ", s).strip()
    if s.lower() in EMPTY_TOKENS:
        return None
    return s

def norm_for_key(x):
    s = clean_str(x)
    if s is None:
        return None
    s = s.lower()
    s = s.replace("·", ".")
    s = re.sub(r"\s+", " ", s)
    s = s.strip()
    return s if s else None

def to_float(x):
    if pd.isna(x):
        return None
    s = str(x)
    m = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if not m:
        return None
    try:
        return float(m[0])
    except Exception:
        return None

def parse_ml_ratio(val):
    """
    Parse M_L_ratio into one float.
      - Numeric like 1.5 -> 1.5
      - Ratio like '1:2' -> 1 / 2
      - Ratio like '1:1:1' -> 1 / (1 + 1)
    """
    if pd.isna(val):
        return None
    s = str(val).strip()
    if clean_str(s) is None:
        return None
    try:
        return float(s)
    except Exception:
        pass

    parts = re.split(r"[:/]", s)
    nums = []
    for p in parts:
        n = to_float(p)
        if n is None:
            return None
        nums.append(n)

    if len(nums) == 1:
        return nums[0]
    metal = nums[0]
    linker_sum = sum(nums[1:])
    if linker_sum == 0:
        return None
    return metal / linker_sum

def parse_year(x):
    y = to_float(x)
    if y is None:
        return None
    y = int(round(y))
    if 1800 <= y <= 2100:
        return y
    return None

def normalize_doi(x):
    s = clean_str(x)
    if s is None:
        return None
    s = s.lower()
    s = re.sub(r"^https?://(dx\.)?doi\.org/", "", s)
    s = re.sub(r"^doi:\s*", "", s)
    s = s.strip()
    return s or None

def display_value(name, abbr=None, include_abbr=False):
    """
    Use full chemical name/formula if available; fall back to abbreviation.
    If include_abbr=True, output 'name (abbr)' when both are available.
    """
    name = clean_str(name)
    abbr = clean_str(abbr)
    if name and abbr and include_abbr and abbr.lower() not in name.lower():
        return f"{name} ({abbr})"
    return name or abbr

def unique_preserve_order(values):
    out = []
    seen = set()
    for v in values:
        v = clean_str(v)
        if not v:
            continue
        k = norm_for_key(v)
        if k not in seen:
            out.append(v)
            seen.add(k)
    return out

def join_with_and(values):
    vals = unique_preserve_order(values)
    if not vals:
        return None
    return " and ".join(vals)

def collect_numbered_reagents(row, stem, max_n=3, include_abbr=False):
    vals = []
    for i in range(1, max_n + 1):
        vals.append(display_value(
            row.get(f"{stem}_{i}"),
            row.get(f"{stem}_{i}_abbr"),
            include_abbr=include_abbr
        ))
    return unique_preserve_order(vals)

def collect_linkers(row):
    return collect_numbered_reagents(row, "linker", max_n=3)

def collect_modulators(row):
    return collect_numbered_reagents(row, "modulator", max_n=2)

def collect_solvents(row):
    vals = [
        display_value(row.get("solvent_main"), row.get("solvent_main_abbr")),
        display_value(row.get("solvent_secondary"), row.get("solvent_secondary_abbr")),
    ]
    return unique_preserve_order(vals)

def canonical_set_key(values):
    vals = [norm_for_key(v) for v in values]
    vals = sorted({v for v in vals if v})
    return " + ".join(vals) if vals else "unknown"

def primary_metal_precursor(row):
    return display_value(row.get("metal_1"), row.get("metal_1_abbr"))

def extract_primary_metal_element(row):
    """
    Best-effort extraction of the metal element from metal_1_abbr or metal_1.
    This is used only if CLUSTER_METAL_MODE = 'element'.
    """
    abbr = clean_str(row.get("metal_1_abbr"))
    if abbr:
        m = re.search(r"\b([A-Z][a-z]?)\b", abbr)
        if m:
            return m.group(1)

    text = clean_str(row.get("metal_1"))
    if not text:
        return "Me_unknown"

    low = text.lower()
    for name, sym in ELEMENT_NAMES.items():
        if re.search(rf"\b{name}\b", low):
            return sym

    text2 = re.sub(r"^[^A-Za-z]+", "", text)
    m = re.match(r"([A-Z][a-z]?)", text2)
    if m:
        return m.group(1)

    m = re.search(r"\b([A-Z][a-z]?)\b", text)
    if m:
        return m.group(1)

    return "Me_unknown"

def build_cluster_key(row):
    """
    Main holdout grouping.
    Default grouping: primary metal precursor + all linkers + all solvents.
    This creates more clusters than metal-element + linker-family and avoids
    forcing MXA and MXB to be in the same split.
    """
    if CLUSTER_METAL_MODE == "element":
        metal_key = norm_for_key(extract_primary_metal_element(row))
    elif CLUSTER_METAL_MODE == "precursor":
        metal_key = norm_for_key(primary_metal_precursor(row))
    else:
        raise ValueError("CLUSTER_METAL_MODE must be 'precursor' or 'element'.")

    parts = [
        f"metal={metal_key or 'unknown'}",
        f"linker={canonical_set_key(collect_linkers(row))}",
        f"solvent={canonical_set_key(collect_solvents(row))}",
    ]

    if INCLUDE_MODULATOR_IN_CLUSTER:
        parts.append(f"modulator={canonical_set_key(collect_modulators(row))}")

    return "|".join(parts)

def row_to_conditions(row):
    return {
        "metal_precursor": primary_metal_precursor(row),
        "organic_linker": join_with_and(collect_linkers(row)),
        "modulator": join_with_and(collect_modulators(row)),
        "solvent": join_with_and(collect_solvents(row)),
        "metal_concentration_mM": to_float(row.get("metel_concnertation")),
        "M_L_ratio": parse_ml_ratio(row.get("M_L_ratio")),
        "temperature_C": to_float(row.get("temperature_c")),
        "time_h": to_float(row.get("time_h")),
    }

def canonical_condition_key(row):
    cond = row_to_conditions(row)
    key = {}
    for k, v in cond.items():
        if isinstance(v, (float, int, np.floating, np.integer)):
            key[k] = None if pd.isna(v) else round(float(v), 8)
        elif v is None:
            key[k] = None
        else:
            key[k] = norm_for_key(v)
    return json.dumps(key, sort_keys=True, ensure_ascii=False)

def to_messages_record(row):
    label = LABEL_POS if bool(row["is_success"]) else LABEL_NEG
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(row_to_conditions(row), ensure_ascii=False)},
            {"role": "assistant", "content": label},
        ]
    }

def count_labels(df):
    p = int(df["is_success"].sum())
    n = int((~df["is_success"]).sum())
    return {LABEL_POS: p, LABEL_NEG: n}

def write_jsonl(df, path, seed=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    out = df
    if SHUFFLE_OUTPUT and len(out) > 1:
        out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    with open(path, "w", encoding="utf-8") as f:
        for _, row in out.iterrows():
            f.write(json.dumps(to_messages_record(row), ensure_ascii=False) + "\n")

# --------------------------
# Metadata
# --------------------------
def load_publication_years(path):
    meta = pd.read_excel(path)
    if "DOI" not in meta.columns or "Publication Year" not in meta.columns:
        raise ValueError("Full.xlsx must contain columns 'DOI' and 'Publication Year'.")

    meta = meta.copy()
    meta["doi_norm"] = meta["DOI"].map(normalize_doi)
    meta["publication_year"] = meta["Publication Year"].map(parse_year)
    meta = meta[meta["doi_norm"].notna() & meta["publication_year"].notna()].copy()

    def choose_year(s):
        vals = [int(x) for x in s.dropna().tolist()]
        if not vals:
            return np.nan
        mode = pd.Series(vals).mode()
        return int(mode.iloc[0])

    return (
        meta.groupby("doi_norm", as_index=False)["publication_year"]
            .agg(choose_year)
    )

# --------------------------
# Holdout cluster choice
# --------------------------
def choose_holdout_clusters(df, cluster_col, holdout_frac, cluster_frac, seed, mode="both", n_trials=800):
    """
    Choose holdout clusters reproducibly while balancing:
      - row count, around holdout_frac of rows
      - label count, around holdout_frac of P and N
      - cluster count, around cluster_frac of clusters when mode is "both" or "clusters"
    """
    stats = (
        df.groupby(cluster_col)
          .agg(n_rows=(cluster_col, "size"), n_pos=("is_success", "sum"))
          .reset_index()
    )
    stats["n_pos"] = stats["n_pos"].astype(int)
    stats["n_neg"] = stats["n_rows"] - stats["n_pos"]

    if len(stats) < 2:
        raise ValueError("Need at least 2 clusters to create train/holdout split.")

    total_rows = len(df)
    total_pos = int(df["is_success"].sum())
    total_neg = int((~df["is_success"]).sum())
    total_clusters = len(stats)

    target_rows = max(1, int(round(total_rows * holdout_frac)))
    target_pos = max(1, int(round(total_pos * holdout_frac)))
    target_neg = max(1, int(round(total_neg * holdout_frac)))
    target_clusters = max(1, int(round(total_clusters * cluster_frac)))

    rng = np.random.default_rng(seed)

    def score(sel_idx):
        sub = stats.iloc[list(sel_idx)] if len(sel_idx) else stats.iloc[[]]
        n_rows = int(sub["n_rows"].sum())
        n_pos = int(sub["n_pos"].sum())
        n_neg = int(sub["n_neg"].sum())
        n_clusters = len(sel_idx)

        def rel(v, t):
            return abs(v - t) / max(1, t)

        # Label balance matters strongly because this is a binary classifier.
        cost = 0.0
        if mode in {"rows", "both"}:
            cost += 1.00 * rel(n_rows, target_rows)
        if mode in {"clusters", "both"}:
            cost += 0.65 * rel(n_clusters, target_clusters)
        cost += 0.85 * rel(n_pos, target_pos)
        cost += 0.85 * rel(n_neg, target_neg)

        # Avoid pathological holdouts with one class nearly absent.
        if n_pos == 0 or n_neg == 0:
            cost += 10.0
        return cost, n_rows, n_pos, n_neg, n_clusters

    # Randomized candidate construction. Prefer many small to medium clusters
    # when mode includes cluster targeting, but still allow larger clusters if
    # needed for row and label balance.
    best_sel = None
    best_score = None
    n = len(stats)

    for trial in range(max(1, int(n_trials))):
        if mode == "clusters":
            desired_k = target_clusters
        elif mode == "rows":
            desired_k = None
        else:
            # Small jitter avoids the same local optimum every trial.
            jitter = rng.integers(-max(1, target_clusters // 20), max(2, target_clusters // 20 + 1))
            desired_k = int(np.clip(target_clusters + jitter, 1, n - 1))

        order = rng.permutation(n)
        sel = []
        cur_rows = cur_pos = cur_neg = 0

        if desired_k is not None:
            # Weighted sampling without replacement. In "both" mode this favors
            # smaller clusters so the requested cluster count is achievable.
            weights = 1.0 / np.sqrt(stats["n_rows"].to_numpy(dtype=float))
            weights = weights / weights.sum()
            sel = rng.choice(np.arange(n), size=desired_k, replace=False, p=weights).tolist()
        else:
            # Row-target mode: greedily add clusters until close to target rows.
            for idx in order:
                r = stats.iloc[int(idx)]
                if cur_rows >= target_rows:
                    break
                sel.append(int(idx))
                cur_rows += int(r["n_rows"])
                cur_pos += int(r["n_pos"])
                cur_neg += int(r["n_neg"])

        # Simple local swaps improve row and label balance.
        sel_set = set(sel)
        not_sel = set(range(n)) - sel_set
        cur_score = score(sel_set)[0]
        for _ in range(300):
            if not sel_set or not not_sel:
                break
            remove_idx = int(rng.choice(list(sel_set)))
            add_idx = int(rng.choice(list(not_sel)))
            trial_set = set(sel_set)
            trial_set.remove(remove_idx)
            trial_set.add(add_idx)
            trial_score = score(trial_set)[0]
            if trial_score < cur_score or rng.random() < 0.003:
                sel_set = trial_set
                not_sel = set(range(n)) - sel_set
                cur_score = trial_score

        sc = score(sel_set)
        if best_score is None or sc[0] < best_score[0]:
            best_score = sc
            best_sel = sel_set

    chosen = set(stats.iloc[list(best_sel)][cluster_col].tolist())
    return chosen

# Backward-compatible alias.
def choose_holdout_clusters_by_rows(df, cluster_col, holdout_frac, seed):
    return choose_holdout_clusters(
        df=df,
        cluster_col=cluster_col,
        holdout_frac=holdout_frac,
        cluster_frac=HOLDOUT_CLUSTER_FRAC,
        seed=seed,
        mode="rows",
        n_trials=HOLDOUT_SEARCH_TRIALS,
    )

# --------------------------
# Year bins inside train
# --------------------------
def make_contiguous_year_bins(train_df, n_bins):
    ydf = train_df[train_df["publication_year"].notna()].copy()
    if ydf.empty:
        return []

    ydf["publication_year"] = ydf["publication_year"].astype(int)
    counts = ydf.groupby("publication_year").size().sort_index()

    years = list(counts.index)
    n_bins = min(n_bins, len(years))
    if n_bins <= 0:
        return []

    total = int(counts.sum())
    cum = counts.cumsum().to_numpy()

    cut_positions = []
    last_cut = -1
    for i in range(1, n_bins):
        target = total * i / n_bins
        min_idx = last_cut + 1
        max_idx = len(years) - (n_bins - i) - 1  # leave at least one year per remaining bin
        idxs = np.arange(min_idx, max_idx + 1)
        chosen = int(idxs[np.argmin(np.abs(cum[idxs] - target))])
        cut_positions.append(chosen)
        last_cut = chosen

    cut_positions.append(len(years) - 1)

    bins = []
    start_idx = 0
    for bin_i, cut_idx in enumerate(cut_positions, start=1):
        start_year = int(years[start_idx])
        end_year = int(years[cut_idx])
        bdf = ydf[
            (ydf["publication_year"] >= start_year)
            & (ydf["publication_year"] <= end_year)
        ].copy()
        bins.append({
            "bin": bin_i,
            "start_year": start_year,
            "end_year": end_year,
            "df": bdf,
        })
        start_idx = cut_idx + 1

    return bins

def year_range_name(start_year, end_year):
    return f"{start_year}" if start_year == end_year else f"{start_year}to{end_year}"

# --------------------------
# Main
# --------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

pos_df = pd.read_csv(POS_PATH, low_memory=False)
neg_df = pd.read_csv(NEG_PATH, low_memory=False)

pos_df["is_success"] = True
neg_df["is_success"] = False

full_df = pd.concat([pos_df, neg_df], ignore_index=True)
n_raw = len(full_df)

# Link DOI to publication year.
year_map = load_publication_years(FULL_METADATA_PATH)
full_df["doi_norm"] = full_df.get("doi", pd.Series([None] * len(full_df))).map(normalize_doi)
full_df = full_df.merge(year_map, on="doi_norm", how="left")

# Required information for this exact classifier input.
def has_any(row, cols):
    return any(clean_str(row.get(c)) is not None for c in cols)

mask = pd.Series(True, index=full_df.index)
mask &= full_df.apply(lambda r: has_any(r, ["metal_1", "metal_1_abbr"]), axis=1)
mask &= full_df.apply(
    lambda r: has_any(r, ["linker_1", "linker_1_abbr", "linker_2", "linker_2_abbr", "linker_3", "linker_3_abbr"]),
    axis=1
)
mask &= full_df.apply(
    lambda r: has_any(r, ["solvent_main", "solvent_main_abbr", "solvent_secondary", "solvent_secondary_abbr"]),
    axis=1
)
mask &= full_df["metel_concnertation"].map(lambda x: to_float(x) is not None)
mask &= full_df["M_L_ratio"].map(lambda x: parse_ml_ratio(x) is not None)

filtered_df = full_df[mask].copy()
n_after_required = len(filtered_df)

# Drop unlearnable contradictions: same model input, both labels.
filtered_df["condition_key"] = filtered_df.apply(canonical_condition_key, axis=1)
label_nunique = filtered_df.groupby("condition_key")["is_success"].nunique()
conflict_keys = set(label_nunique[label_nunique > 1].index)
n_conflict_rows = int(filtered_df["condition_key"].isin(conflict_keys).sum())

if DROP_INPUT_LABEL_CONFLICTS and conflict_keys:
    filtered_df = filtered_df[~filtered_df["condition_key"].isin(conflict_keys)].copy()

n_after_conflict = len(filtered_df)

n_deduped_rows = 0
if DEDUP_EXACT_INPUT_WITHIN_LABEL:
    before = len(filtered_df)
    filtered_df = filtered_df.drop_duplicates(subset=["condition_key", "is_success"]).copy()
    n_deduped_rows = before - len(filtered_df)

# Main split by primary metal + all linkers + all solvents.
filtered_df["cluster_key"] = filtered_df.apply(build_cluster_key, axis=1)

holdout_clusters = choose_holdout_clusters(
    filtered_df,
    cluster_col="cluster_key",
    holdout_frac=HOLDOUT_FRAC,
    cluster_frac=HOLDOUT_CLUSTER_FRAC,
    seed=RNG_SEED,
    mode=HOLDOUT_TARGET_MODE,
    n_trials=HOLDOUT_SEARCH_TRIALS,
)

filtered_df["is_holdout"] = filtered_df["cluster_key"].isin(holdout_clusters)
train_df = filtered_df[~filtered_df["is_holdout"]].copy()
holdout_df = filtered_df[filtered_df["is_holdout"]].copy()

# Write main split.
write_jsonl(train_df, TRAIN_OUT, seed=RNG_SEED + 1)
write_jsonl(holdout_df, HOLDOUT_OUT, seed=RNG_SEED + 2)

with open(CLASS_MAP_OUT, "w", encoding="utf-8") as f:
    json.dump({"P": "success", "N": "failure"}, f, ensure_ascii=False, indent=2)

# Write year bins and cumulative year files within train only.
year_outputs = []
bins = make_contiguous_year_bins(train_df, YEAR_BINS)

for b in bins:
    name = year_range_name(b["start_year"], b["end_year"])
    path = OUT_DIR / f"mof_cls_train_{name}.jsonl"
    write_jsonl(b["df"], path, seed=RNG_SEED + 100 + b["bin"])
    year_outputs.append({
        "type": "single_bin",
        "bin": int(b["bin"]),
        "path": str(path),
        "start_year": int(b["start_year"]),
        "end_year": int(b["end_year"]),
        "rows": int(len(b["df"])),
        "labels": count_labels(b["df"]),
    })

# Cumulative 1+2 and 1+2+3 only. No 1 because it is the first bin;
# no 1+2+3+4 because it is essentially full train for rows with known year.
for upto in [2, 3]:
    if len(bins) >= upto:
        cum_df = pd.concat([b["df"] for b in bins[:upto]], ignore_index=False)
        start_year = int(bins[0]["start_year"])
        end_year = int(bins[upto - 1]["end_year"])
        name = year_range_name(start_year, end_year)
        path = OUT_DIR / f"mof_cls_train_{name}.jsonl"
        write_jsonl(cum_df, path, seed=RNG_SEED + 200 + upto)
        year_outputs.append({
            "type": f"cumulative_1to{upto}",
            "path": str(path),
            "start_year": start_year,
            "end_year": end_year,
            "rows": int(len(cum_df)),
            "labels": count_labels(cum_df),
        })

summary = {
    "config": {
        "rng_seed": RNG_SEED,
        "holdout_frac": HOLDOUT_FRAC,
        "holdout_cluster_frac": HOLDOUT_CLUSTER_FRAC,
        "holdout_target_mode": HOLDOUT_TARGET_MODE,
        "holdout_search_trials": HOLDOUT_SEARCH_TRIALS,
        "cluster_metal_mode": CLUSTER_METAL_MODE,
        "cluster_definition": "primary metal + all linkers + all solvents"
                              + (" + all modulators" if INCLUDE_MODULATOR_IN_CLUSTER else ""),
        "drop_input_label_conflicts": DROP_INPUT_LABEL_CONFLICTS,
        "dedup_exact_input_within_label": DEDUP_EXACT_INPUT_WITHIN_LABEL,
    },
    "counts": {
        "input_rows_total": int(n_raw),
        "rows_after_required_field_checks": int(n_after_required),
        "rows_skipped_required": int(n_raw - n_after_required),
        "rows_with_conflicting_input_labels": int(n_conflict_rows),
        "rows_after_conflict_filter": int(n_after_conflict),
        "rows_deduped_exact_input_within_label": int(n_deduped_rows),
        "rows_final": int(len(filtered_df)),
        "rows_with_publication_year_final": int(filtered_df["publication_year"].notna().sum()),
        "train_rows": int(len(train_df)),
        "holdout_rows": int(len(holdout_df)),
        "train_missing_publication_year_rows": int(train_df["publication_year"].isna().sum()),
        "holdout_missing_publication_year_rows": int(holdout_df["publication_year"].isna().sum()),
    },
    "labels": {
        "train": count_labels(train_df),
        "holdout": count_labels(holdout_df),
    },
    "clusters": {
        "unique_clusters_final": int(filtered_df["cluster_key"].nunique()),
        "holdout_clusters": int(len(holdout_clusters)),
        "train_clusters": int(train_df["cluster_key"].nunique()),
    },
    "outputs": {
        "train": str(TRAIN_OUT),
        "holdout": str(HOLDOUT_OUT),
        "class_map": str(CLASS_MAP_OUT),
        "year_outputs": year_outputs,
    },
}

with open(SUMMARY_OUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# --------------------------
# Report
# --------------------------
print("=== Summary ===")
print(f"Input rows total: {n_raw}")
print(f"Rows kept after required field checks: {n_after_required}")
print(f"Rows skipped required: {n_raw - n_after_required}")
print(f"Rows with same input JSON but both P and N: {n_conflict_rows}")
if DROP_INPUT_LABEL_CONFLICTS:
    print(f"Rows after conflict drop: {n_after_conflict}")
if DEDUP_EXACT_INPUT_WITHIN_LABEL:
    print(f"Rows deduped within label: {n_deduped_rows}")
print()
print("Cluster split:")
print(f"  Cluster key: primary metal + all linkers + all solvents")
print(f"  Unique clusters: {filtered_df['cluster_key'].nunique()}")
print(f"  Holdout clusters: {len(holdout_clusters)} ({len(holdout_clusters) / max(1, filtered_df['cluster_key'].nunique()) * 100:.2f}% of clusters)")
print(f"  Train rows: {len(train_df)} labels={count_labels(train_df)}")
print(f"  Holdout rows: {len(holdout_df)} ({len(holdout_df) / max(1, len(filtered_df)) * 100:.2f}% of rows) labels={count_labels(holdout_df)}")
print()
print("Publication year:")
print(f"  Final rows with publication year: {filtered_df['publication_year'].notna().sum()} / {len(filtered_df)}")
print(f"  Train rows missing publication year, included in full train but not year subsets: {train_df['publication_year'].isna().sum()}")
print()
print("Year subset files:")
for y in year_outputs:
    print(f"  {y['type']}: {Path(y['path']).name}, years {y['start_year']}-{y['end_year']}, rows={y['rows']}, labels={y['labels']}")
print()
print("Wrote files:")
print(f"  {TRAIN_OUT}")
print(f"  {HOLDOUT_OUT}")
print(f"  {CLASS_MAP_OUT}")
print(f"  {SUMMARY_OUT}")
for y in year_outputs:
    print(f"  {y['path']}")

print("\nSample train records:")
with open(TRAIN_OUT, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        print(line.strip())


=== Summary ===
Input rows total: 37878
Rows kept after required field checks: 34419
Rows skipped required: 3459
Rows with same input JSON but both P and N: 2877
Rows after conflict drop: 31542

Cluster split:
  Cluster key: primary metal + all linkers + all solvents
  Unique clusters: 10291
  Holdout clusters: 1060 (10.30% of clusters)
  Train rows: 28388 labels={'P': 11854, 'N': 16534}
  Holdout rows: 3154 (10.00% of rows) labels={'P': 1317, 'N': 1837}

Publication year:
  Final rows with publication year: 31541 / 31542
  Train rows missing publication year, included in full train but not year subsets: 1

Year subset files:
  single_bin: mof_cls_train_1999to2012.jsonl, years 1999-2012, rows=6521, labels={'P': 2571, 'N': 3950}
  single_bin: mof_cls_train_2013to2016.jsonl, years 2013-2016, rows=7679, labels={'P': 3094, 'N': 4585}
  single_bin: mof_cls_train_2017to2020.jsonl, years 2017-2020, rows=7547, labels={'P': 3081, 'N': 4466}
  single_bin: mof_cls_train_2021to2025.jsonl, years 20

## 2. Optional: Estimate fine-tuning token counts and approximate cost

This optional section estimates token counts for the generated JSONL files. Update model names and pricing assumptions as needed.


In [2]:
# Estimate fine-tuning tokens and cost for the JSONL files produced by the first cell.
# Edit MODEL, NUM_EPOCHS, and PRICES_USD_PER_M_TOKENS as needed.

import json, sys, subprocess, math, os
from pathlib import Path
from statistics import mean, median
import numpy as np

# Try to import tiktoken, install if missing
try:
    import tiktoken
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken", "-qqq"])
    import tiktoken

# -----------------------
# Config
# -----------------------
DATA_DIR = Path("Expected Output")
FILES = {
    "train": DATA_DIR / "mof_cls_train.jsonl",
    "holdout": DATA_DIR / "mof_cls_holdout.jsonl",
    "year-wise": DATA_DIR / "mof_cls_train_1999to2016.jsonl",
}

# Choose a base model id that matches what you plan to fine-tune
MODEL = "gpt-4.1-mini-2025-04-14"   # or "gpt-4.1-2025-04-14", "gpt-4o-mini-2024-07-18", etc.
NUM_EPOCHS = 3                       # change to your planned epochs

# USD per 1M training tokens (edit if pricing changes)
PRICES_USD_PER_M_TOKENS = {
    # From OpenAI pricing page at time of writing
    "gpt-4.1-2025-04-14": 25.00,
    "gpt-4.1-mini-2025-04-14": 5.00,
    "gpt-4.1-nano-2025-04-14": 1.50,
    "gpt-4o-2024-08-06": 25.00,
    "gpt-4o-mini-2024-07-18": 3.00,
}
# Fallback price if your model key is not in the table
DEFAULT_TRAIN_PRICE = 25.00

# -----------------------
# Token counting helpers
# -----------------------
SUPPORTED_MODELS_FOR_ENCODING = {
    # Map common aliases to encodings, with graceful fallback
    "gpt-4.1-2025-04-14": "o200k_base",
    "gpt-4.1-mini-2025-04-14": "o200k_base",
    "gpt-4.1-nano-2025-04-14": "o200k_base",
    "gpt-4o-2024-08-06": "o200k_base",
    "gpt-4o-mini-2024-07-18": "o200k_base",
}

def get_encoding_for_model(model: str):
    # Try model specific encoding first
    try:
        return tiktoken.encoding_for_model(model)
    except Exception:
        # Fallback to a best-effort encoding
        name = SUPPORTED_MODELS_FOR_ENCODING.get(model, "o200k_base")
        return tiktoken.get_encoding(name)

def num_tokens_from_messages(messages, model: str) -> int:
    """
    Estimate tokens for a list of chat messages for fine-tuning.
    Rule-of-thumb:
      - 3 special tokens per message
      - +1 extra token for assistant messages
      - plus tokens for 'role' and 'content'
    """
    encoding = get_encoding_for_model(model)
    tokens_per_message = 3
    total = 0
    for msg in messages:
        role = str(msg.get("role", ""))
        content = str(msg.get("content", ""))
        total += tokens_per_message
        if role == "assistant":
            total += 1  # assistant extra token
        total += len(encoding.encode(role))
        total += len(encoding.encode(content))
    return total

def iter_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def file_token_stats(path: Path, model: str):
    if not path.exists():
        return {"exists": False}
    counts = []
    n = 0
    for rec in iter_jsonl(path):
        msgs = rec.get("messages", [])
        counts.append(num_tokens_from_messages(msgs, model))
        n += 1
    if n == 0:
        return {"exists": True, "records": 0, "total_tokens": 0, "avg": 0, "p5": 0, "p50": 0, "p95": 0}
    arr = np.array(counts, dtype=np.int64)
    return {
        "exists": True,
        "records": int(n),
        "total_tokens": int(arr.sum()),
        "avg": float(arr.mean()),
        "p5": float(np.percentile(arr, 5)),
        "p50": float(np.percentile(arr, 50)),
        "p95": float(np.percentile(arr, 95)),
    }

def usd_cost_from_tokens(total_tokens: int, price_per_million: float) -> float:
    return (total_tokens / 1_000_000.0) * price_per_million

# -----------------------
# Run
# -----------------------
price = PRICES_USD_PER_M_TOKENS.get(MODEL, DEFAULT_TRAIN_PRICE)
print(f"Model: {MODEL}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Training price: ${price:.2f} per 1M tokens\n")

stats = {}
for name, path in FILES.items():
    stats[name] = file_token_stats(path, MODEL)

# Report per file
for name, st in stats.items():
    if not st.get("exists"):
        print(f"{name}: file not found -> {FILES[name]}")
        continue
    print(f"{name}:")
    print(f"  path: {FILES[name]}")
    print(f"  records: {st['records']:,}")
    print(f"  total tokens (per epoch): {st['total_tokens']:,}")
    print(f"  avg tokens per record: {st['avg']:.1f}")
    print(f"  p5, p50, p95 tokens per record: {st['p5']:.0f}, {st['p50']:.0f}, {st['p95']:.0f}")
    print()

# Training cost is based on the training split only
train_tokens_one_epoch = stats.get("train", {}).get("total_tokens", 0)
total_training_tokens = train_tokens_one_epoch * NUM_EPOCHS
est_cost_usd = usd_cost_from_tokens(total_training_tokens, price)

print("Summary:")
print(f"  training tokens per epoch: {train_tokens_one_epoch:,}")
print(f"  total training tokens ({NUM_EPOCHS} epochs): {total_training_tokens:,}")
print(f"  estimated training cost: ${est_cost_usd:,.2f}")

# Optional: show what it would cost if you trained on pos_only instead
pos_only_tokens_one_epoch = stats.get("pos_only", {}).get("total_tokens", 0)
pos_only_total_tokens = pos_only_tokens_one_epoch * NUM_EPOCHS
pos_only_cost = usd_cost_from_tokens(pos_only_total_tokens, price)
print("\nIf training on pos_only instead:")
print(f"  tokens per epoch: {pos_only_tokens_one_epoch:,}")
print(f"  total tokens: {pos_only_total_tokens:,}")
print(f"  estimated training cost: ${pos_only_cost:,.2f}")


Model: gpt-4.1-mini-2025-04-14
Epochs: 3
Training price: $5.00 per 1M tokens

train:
  path: Expected Output\mof_cls_train.jsonl
  records: 28,388
  total tokens (per epoch): 5,920,622
  avg tokens per record: 208.6
  p5, p50, p95 tokens per record: 192, 206, 230

holdout:
  path: Expected Output\mof_cls_holdout.jsonl
  records: 3,154
  total tokens (per epoch): 657,740
  avg tokens per record: 208.5
  p5, p50, p95 tokens per record: 194, 204, 229

year-wise:
  path: Expected Output\mof_cls_train_1999to2016.jsonl
  records: 14,200
  total tokens (per epoch): 2,966,367
  avg tokens per record: 208.9
  p5, p50, p95 tokens per record: 193, 207, 231

Summary:
  training tokens per epoch: 5,920,622
  total training tokens (3 epochs): 17,761,866
  estimated training cost: $88.81

If training on pos_only instead:
  tokens per epoch: 0
  total tokens: 0
  estimated training cost: $0.00
